
# LoRA vs. QLoRA on Legal Text

**Day 2 — Tools & LLM Finetuning · Practical 1 of 3 · Companion to the "LLM Finetuning" deck**

> **Running in Google Colab:** requires an **A100 (40GB) GPU runtime** (Colab Pro: Runtime ->
> Change runtime type -> A100 GPU). Llama-3.1-8B-Instruct's plain-LoRA path alone needs ~16GB
> just for bf16 weights, plus activations and optimizer state on top -- it will not fit a
> free-tier T4 (~15GB). The QLoRA path (4-bit base) is far lighter and would fit smaller GPUs,
> but since both paths run in the same session for direct comparison, A100 is required here.

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Finetune a language model on real legal-clause text using **plain LoRA**
2. Finetune the SAME model using **QLoRA** (4-bit quantized base + LoRA adapters)
3. Compare trainable parameter counts, VRAM usage, and generation quality side by side
4. Connect what you observe directly back to the deck's math: `W' = W + (alpha/r) * B * A`,
   and the "QLoRA solves a different problem than LoRA" argument

## Why This Matters for a Law Firm

Day 1 finetuned DistilGPT2 with **full finetuning** — every weight updated. That doesn't scale
to real production-size models. This notebook makes LoRA and QLoRA's trade-offs concrete: you
will watch the trainable-parameter count drop by orders of magnitude, and watch VRAM usage drop
further still once the base model itself is quantized.

## Notebook Workflow

```mermaid
flowchart TD
    A["Base model\n(Llama-3.1-8B-Instruct)"] --> B["Real legal clause\ndataset (LEDGAR)"]
    B --> C["Path 1: Plain LoRA\n(bf16 base + LoRA adapters)"]
    B --> D["Path 2: QLoRA\n(4-bit base + LoRA adapters)"]
    C --> E["Compare: trainable params,\nVRAM, generation quality"]
    D --> E



## Section 1 — Setup

We use **Llama-3.1-8B-Instruct** instead of a smaller model. An earlier version of this
notebook used Qwen2.5-1.5B-Instruct, whose pretrain corpus is roughly balanced between English
and Chinese -- when a small, under-trained LoRA adapter combined with an unguarded sampling
config (no repetition penalty) caused degenerate generation, the output drifted into Chinese
tokens rather than staying in English gibberish. Llama-3.1's pretrain corpus is overwhelmingly
English, which removes that specific failure mode. Both the dataset (Section 2) and the
generation config (Section 7) are also fixed below -- the language drift had three compounding
causes, not just model choice.

At 8B parameters, training is **step-capped, not epoch-capped**: Sections 4 and 6 each train
for a fixed `max_steps` calibrated to fit a 30-minute combined budget on an A100, printing
wall-clock time so you can raise or lower it on a re-run based on your actual throughput.


In [ ]:

%pip install -q -U transformers accelerate peft bitsandbytes datasets trl huggingface_hub

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:

import os
from huggingface_hub import login

def get_hf_token():
    # In Google Colab, prefer the Secrets manager (key icon in the left sidebar) --
    # keeps the token out of the notebook file. Falls back to a plain env var locally.
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except ImportError:
        pass
    return os.environ.get("HF_TOKEN")


hf_token = get_hf_token()
if not hf_token:
    raise ValueError(
        "Set an HF_TOKEN secret (Colab key icon) with access to "
        "meta-llama/Llama-3.1-8B-Instruct (accept the license on huggingface.co first)."
    )
login(token=hf_token)
print("Logged in to Hugging Face Hub.")



## Section 2 — A Real Legal Finetuning Dataset

**LEDGAR** (Labeled EDGAR): contract provisions extracted from SEC EDGAR filings, part of the
LexGLUE benchmark. The full corpus has ~846k clauses across 12.6k clause types; we sample
15,000 shuffled provisions here. Training itself (Sections 4 and 6) is step-capped rather than
epoch-capped, so this pool size controls sampling diversity, not training time.


In [ ]:

from datasets import load_dataset

SUBSET_SIZE = 15000  # pool size to sample from; training is step-capped below, so raising
                      # this increases sampling diversity, not training time.

ledgar = load_dataset("coastalcph/lex_glue", "ledgar", split="train")
print(f"Full LEDGAR train split: {len(ledgar):,} contract provisions")

raw_dataset = ledgar.shuffle(seed=42).select(range(SUBSET_SIZE)).remove_columns(
    [c for c in ledgar.column_names if c != "text"]
)
print(f"Sampled for this notebook: {len(raw_dataset):,} examples")
print("\nSample entries:")
for i in range(3):
    text = raw_dataset[i]["text"]
    print(f"\n  [{i}] {text[:200]}{'...' if len(text) > 200 else ''}")



## Section 3 — Path 1: Plain LoRA

`peft`'s `LoraConfig` freezes the base model and injects trainable low-rank adapters into the
attention projection layers -- exactly the `W' = W + (alpha/r) * B * A` mechanism from the deck.
The base model here loads in `bfloat16` (half precision), NOT quantized.


In [ ]:

from peft import LoraConfig, get_peft_model

MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

lora_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

lora_config = LoraConfig(
    r=16,                                                          # rank -- controls capacity/cost trade-off
    lora_alpha=32,                                                 # scaling factor (alpha/r in the deck's formula)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],       # inject LoRA into all attention projections
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

lora_model = get_peft_model(lora_base_model, lora_config)
lora_model.print_trainable_parameters()



The output above is the headline number: compare "trainable params" to "all params" -- this is
the deck's "100-1000x fewer trainable parameters than full finetuning" claim, made concrete on
a real model.



## Section 4 — Train the LoRA Model

A short training run using TRL's `SFTTrainer`, the same trainer covered in the Tooling &
Frameworks deck.


In [ ]:

from trl import SFTTrainer, SFTConfig
import time

# Wrap each LEDGAR clause as an instruct-style example: the model is prompted to draft a
# clause, and the real clause text is the target completion. Llama-3.1-8B-Instruct was
# instruction-tuned on this chat format, so training on it (rather than bare text) keeps
# the model on-distribution.
def to_chat_text(example):
    messages = [
        {"role": "user", "content": "Draft a commercial contract clause."},
        {"role": "assistant", "content": example["text"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

lora_chat_dataset = raw_dataset.map(to_chat_text)

lora_training_args = SFTConfig(
    output_dir="./llama-lora-legal",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    max_steps=300,            # time-capped, not epoch-capped -- see Section 1 note.
                               # packing=False below means fewer tokens/step than a packed
                               # run; adjust based on the wall-clock printed after training.
    learning_rate=1e-4,       # aggressive LR (2e-4) + no warmup on an 8B model with r=16
                               # destabilized training in an earlier version of this notebook
                               # and produced garbled, non-English output.
    warmup_steps=9,       # ~3% of max_steps=300; warmup_ratio is deprecated in newer transformers
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    max_length=256,
    packing=False,
    loss_type="nll",     # TRL's default "chunked_nll" patches model.forward for
                          # memory savings, but that patch is incompatible with a
                          # PEFT-wrapped forward (functools.partial has no __func__),
                          # raising AttributeError inside SFTTrainer's init. Plain
                          # "nll" skips that patch entirely.            # packing concatenates multiple LEDGAR clauses into one block
                               # with no boundary-aware loss masking -- the model ends up
                               # training on clause-A-bleeding-into-clause-B sequences. This
                               # was the main cause of word-salad output in an earlier run.
    bf16=True,
)

lora_trainer = SFTTrainer(
    model=lora_model,
    train_dataset=lora_chat_dataset,
    args=lora_training_args,
)

_start = time.perf_counter()
lora_train_result = lora_trainer.train()
_elapsed = time.perf_counter() - _start
print("\nFinal LoRA training loss:", lora_train_result.training_loss)
print(f"LoRA training wall-clock: {_elapsed / 60:.1f} min")
print("Well under ~12 min? Raise max_steps and re-run for a better-trained model.")
print("Over ~15 min? Lower max_steps -- the QLoRA path below still needs to run.")

lora_peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak VRAM during LoRA training: {lora_peak_vram_gb:.2f} GB")

# Generate now, before the model is freed below -- this is the only chance to capture a
# genuine LoRA-path completion for the side-by-side comparison in Section 7.
def generate(model, tokenizer, prompt, max_new_tokens=40):
    messages = [{"role": "user", "content": prompt}]
    chat_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            top_k=50,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

lora_completion = generate(lora_model, tokenizer, "Draft a commercial contract clause about")
print("\nLoRA model completion (captured now, before this model is freed):\n")
print(lora_completion)

# Save the LoRA adapter to disk now -- lora_model is freed for VRAM in the next
# section, so this is the only chance to keep it for the Hub-push cells at the end.
lora_model.save_pretrained("./llama-lora-legal-adapter")
print("\nSaved LoRA adapter to ./llama-lora-legal-adapter")



## Section 5 — Path 2: QLoRA

Now the same model, the same LoRA configuration, but with the base model loaded in **4-bit**
via `bitsandbytes` (NF4 quantization, exactly as covered in the deck). We reset CUDA's peak
memory counter first so the two paths' VRAM numbers are directly comparable.


In [ ]:

from transformers import BitsAndBytesConfig

# Free the previous model and reset GPU memory tracking before loading the QLoRA path
del lora_base_model, lora_model, lora_trainer
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # NormalFloat4 -- tailored to the normal weight distribution
    bnb_4bit_compute_dtype=torch.bfloat16, # compute happens in bf16 even though storage is 4-bit
    bnb_4bit_use_double_quant=True,        # Double Quantization -- quantize the quant constants too
)

qlora_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

qlora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

qlora_model = get_peft_model(qlora_base_model, qlora_config)
qlora_model.print_trainable_parameters()



## Section 6 — Train the QLoRA Model

Same training setup as Path 1 -- only the base model's precision changed.


In [ ]:

qlora_chat_dataset = raw_dataset.map(to_chat_text)

qlora_training_args = SFTConfig(
    output_dir="./llama-qlora-legal",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    max_steps=300,            # same time-capped approach as the LoRA path
    learning_rate=1e-4,
    warmup_steps=9,       # ~3% of max_steps=300; warmup_ratio is deprecated in newer transformers
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_strategy="no",
    report_to=[],
    max_length=256,
    packing=False,
    loss_type="nll",     # TRL's default "chunked_nll" patches model.forward for
                          # memory savings, but that patch is incompatible with a
                          # PEFT-wrapped forward (functools.partial has no __func__),
                          # raising AttributeError inside SFTTrainer's init. Plain
                          # "nll" skips that patch entirely.            # same fix as the LoRA path -- see comment there
    bf16=True,
)

qlora_trainer = SFTTrainer(
    model=qlora_model,
    train_dataset=qlora_chat_dataset,
    args=qlora_training_args,
)

_start = time.perf_counter()
qlora_train_result = qlora_trainer.train()
_elapsed = time.perf_counter() - _start
print("\nFinal QLoRA training loss:", qlora_train_result.training_loss)
print(f"QLoRA training wall-clock: {_elapsed / 60:.1f} min")

qlora_peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9
print(f"Peak VRAM during QLoRA training: {qlora_peak_vram_gb:.2f} GB")



## Section 7 — Side-by-Side Comparison

Compare VRAM usage directly, then generate from both finetuned models on the same prompt.


In [ ]:

print("VRAM COMPARISON")
print("-" * 40)
print(f"  Plain LoRA peak VRAM:  {lora_peak_vram_gb:.2f} GB")
print(f"  QLoRA peak VRAM:       {qlora_peak_vram_gb:.2f} GB")
print(f"  Reduction:             {(1 - qlora_peak_vram_gb / lora_peak_vram_gb) * 100:.1f}%")


In [ ]:

prompt = "Draft a commercial contract clause about"

qlora_completion = generate(qlora_model, tokenizer, prompt)

print("LoRA model completion (captured right after training, before it was freed):\n")
print(lora_completion)
print("\n" + "-" * 60 + "\n")
print("QLoRA model completion:\n")
print(qlora_completion)



## Key Takeaways

1. **The trainable-parameter count** printed in Sections 3 and 5 is the direct, visible proof
   of the deck's core LoRA claim -- a tiny fraction of the full model's parameters actually get
   updated, in both paths.
2. **QLoRA's VRAM reduction** comes specifically from the base model's 4-bit storage -- the LoRA
   adapters themselves are the same size in both paths, since only the FROZEN base changed. See
   the printed comparison in Section 7 for this run's actual numbers.
3. **Both paths solve different problems**: LoRA already made the trainable-parameter count
   small; QLoRA additionally makes the base model itself small enough to load on limited
   hardware -- exactly the deck's "QLoRA solves a different problem than LoRA" point.
4. **On model/data/training choice:** an earlier version of this notebook used a much smaller
   model (Qwen2.5-1.5B-Instruct), 12 hardcoded sentences, packed training, and no learning-rate
   warmup -- and its output degenerated into repeated non-English tokens, then later into
   word-salad garbage. None of that was inherent to LoRA/QLoRA -- it came from an under-trained
   adapter on a tiny toy dataset, cross-document bleeding from packing, an LR too aggressive for
   an 8B model with no warmup, and a generation config with no repetition guard. This version
   fixes all of it: a real 15k-sample legal dataset (LEDGAR), an English-dominant 8B base model,
   unpacked training with warmup + cosine decay, and a generation config with repetition
   controls -- fixing only one of these would not have been reliable.

**Next up:** the *Quantization for Inference* notebook -- benchmarking a pre-quantized model
for serving, not training.



## Section 8 — Push Finetuned Adapters to Hugging Face Hub

Both paths trained a **LoRA adapter**, not a full copy of Llama-3.1-8B -- a few hundred MB,
not ~16GB. `push_to_hub()` uploads just the adapter weights + config; anyone loading it later
needs the same base model (`meta-llama/Llama-3.1-8B-Instruct`) plus this adapter on top via
`peft`. Uses the same `hf_token` already authenticated in Section 1 -- requires **write**
access on that token (a read-only token, e.g. one only granted for the gated Llama download,
will fail here with a 403).

Set `HF_USERNAME` below to your Hugging Face username before running.


In [ ]:

HF_USERNAME = "your-hf-username"  # <-- set this to your actual Hugging Face username

lora_repo_id = f"{HF_USERNAME}/llama-3.1-8b-lora-legal-clauses"
qlora_repo_id = f"{HF_USERNAME}/llama-3.1-8b-qlora-legal-clauses"

# --- Push the LoRA adapter (reload from the disk save in Section 4, since lora_model
#     itself was freed from memory before the QLoRA path trained) ---
from peft import PeftModel

lora_base_for_push = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto",
)
lora_adapter_for_push = PeftModel.from_pretrained(lora_base_for_push, "./llama-lora-legal-adapter")
lora_adapter_for_push.push_to_hub(lora_repo_id, token=hf_token)
print(f"LoRA adapter pushed: https://huggingface.co/{lora_repo_id}")

del lora_base_for_push, lora_adapter_for_push
torch.cuda.empty_cache()

# --- Push the QLoRA adapter (still in memory as qlora_model) ---
qlora_model.push_to_hub(qlora_repo_id, token=hf_token)
print(f"QLoRA adapter pushed: https://huggingface.co/{qlora_repo_id}")

# --- To load either adapter later, anywhere (Colab, local machine, production) ---
# from peft import PeftModel
# from transformers import AutoModelForCausalLM, AutoTokenizer
# base = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", torch_dtype=torch.bfloat16)
# model = PeftModel.from_pretrained(base, "your-hf-username/llama-3.1-8b-lora-legal-clauses")
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
